## Paso 1 — Cargamos las 18 tablas ya tipadas

Leemos directo los Parquet generados en `00_setup_y_carga.ipynb` (ya sin
fila fantasma, con piso-sicetac separado en núcleo/raras, tipados y en
camelCase) — sin depender de que ese notebook se haya corrido antes en esta
misma sesión. Esto es lo único que este notebook necesita del anterior.


In [1]:
import pandas as pd
from pathlib import Path

RUTA_INTERIM = Path("../datos/interim")
ARCHIVOS_PARQUET = sorted(RUTA_INTERIM.glob("*.parquet"))

tablas = {archivo.stem: pd.read_parquet(archivo) for archivo in ARCHIVOS_PARQUET}

assert len(tablas) == 18, f"Se esperaban 18 tablas, se cargaron {len(tablas)}"
print(f"Tablas cargadas: {len(tablas)}")
for nombre, df in tablas.items():
    print(f"  {nombre}: {df.shape[0]:>6,} filas x {df.shape[1]:>2} columnas")


Tablas cargadas: 18
  audit_log:     72 filas x  6 columnas
  catalogo-clientes-BUT000-organizaciones: 38,995 filas x 10 columnas
  catalogo_clientes:     21 filas x  6 columnas
  catalogo_configuraciones_vehiculo:     16 filas x  6 columnas
  catalogo_rutas:     31 filas x  5 columnas
  catalogo_servicios_conexos:     11 filas x  8 columnas
  catalogo_usuarios:     11 filas x  4 columnas
  cotizacion_conexos:      7 filas x  5 columnas
  cotizacion_lineas:     48 filas x  9 columnas
  cotizaciones:     18 filas x  9 columnas
  historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy: 25,318 filas x 20 columnas
  inconsistencias:      1 filas x  3 columnas
  lineas_solicitud:     43 filas x 18 columnas
  piso-sicetac_nucleo:    126 filas x 16 columnas
  piso-sicetac_raras:     25 filas x 16 columnas
  solicitudes:     13 filas x 12 columnas
  tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy: 23,020 filas x 24 columnas
  tarifas_historicas:     15 filas x  7 columnas


## Paso 2 — Dimensiones y peso en memoria

Filas, columnas y memoria RAM real que ocupa cada una de las 18 tablas ya
cargadas — para tener una idea de tamaño relativo antes de perfilar columna
por columna.


In [2]:
resumen_tablas = []
for nombre, df in tablas.items():
    memoria_kb = df.memory_usage(deep=True).sum() / 1024
    resumen_tablas.append({
        "tabla": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "memoria_kb": round(memoria_kb, 1),
    })

df_resumen_tablas = pd.DataFrame(resumen_tablas).sort_values("memoria_kb", ascending=False).reset_index(drop=True)
df_resumen_tablas


,tabla,filas,columnas,memoria_kb
0,tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy,23020,24,7153.1
1,historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy,25318,20,5991.4
2,catalogo-clientes-BUT000-organizaciones,38995,10,5085.1
3,piso-sicetac_nucleo,126,16,28.1
4,audit_log,72,6,12.7
5,lineas_solicitud,43,18,10.4
6,cotizacion_lineas,48,9,9.0
7,piso-sicetac_raras,25,16,6.4
8,cotizaciones,18,9,3.2
9,catalogo_rutas,31,5,3.1


## Paso 3 — Perfil columna por columna (18 tablas)

Por cada columna: tipo real, nulos, % de nulos, valores únicos, cardinalidad
relativa (únicos / filas — cerca de 1 sugiere un identificador; cerca de 0
sugiere una categoría con pocas opciones), el valor más frecuente y qué
porcentaje de las filas lo tienen (para detectar columnas cuasi-constantes,
no solo constantes puras), y un ejemplo real.


In [3]:
filas_perfil = []
for nombre, df in tablas.items():
    n = len(df)
    for columna in df.columns:
        serie = df[columna]
        no_nulos = serie.dropna()
        valor_frecuente = no_nulos.mode().iloc[0] if not no_nulos.empty else None
        pct_valor_frecuente = (no_nulos == valor_frecuente).mean() * 100 if not no_nulos.empty else 0.0
        filas_perfil.append({
            "tabla": nombre,
            "columna": columna,
            "dtype": str(serie.dtype),
            "nulos": int(serie.isna().sum()),
            "pct_nulos": round(serie.isna().mean() * 100, 1),
            "unicos": int(no_nulos.nunique()),
            "cardinalidad_relativa": round(no_nulos.nunique() / n, 3) if n else 0.0,
            "valor_mas_frecuente": valor_frecuente,
            "pct_valor_mas_frecuente": round(pct_valor_frecuente, 1),
            "ejemplo": no_nulos.iloc[0] if not no_nulos.empty else None,
        })

df_perfil_columnas = pd.DataFrame(filas_perfil)
print(f"Columnas perfiladas: {len(df_perfil_columnas)}")
df_perfil_columnas


Columnas perfiladas: 184


,tabla,columna,dtype,nulos,pct_nulos,unicos,cardinalidad_relativa,valor_mas_frecuente,pct_valor_mas_frecuente,ejemplo
0,audit_log,id,str,0,0.0,72,1.000,03930948-029C-4B56-A620-54FE98B613FC,1.4,50A6E1BF-E7F8-42D8-9D40-002FA749B109
1,audit_log,usuarioid,str,0,0.0,1,0.014,00000000-0000-0000-0000-000000000001,100.0,00000000-0000-0000-0000-000000000001
2,audit_log,entidad,str,0,0.0,4,0.056,cotizaciones,59.7,cotizaciones
3,audit_log,entidadid,str,0,0.0,39,0.542,0EE0943A-81F2-49F0-A6B8-AF5B2B04FA51,11.1,D782291B-35C9-4532-B845-4A765C1CFF3C
4,audit_log,accion,str,0,0.0,12,0.167,crear,23.6,entregar
...,...,...,...,...,...,...,...,...,...,...
179,tarifas_historicas,configvehiculoid,str,11,73.3,4,0.267,03D0965A-74D8-49FE-8C0E-9861F6306E0A,25.0,8D185987-45FD-409F-B656-D432E77EED28
180,tarifas_historicas,tipologia,str,12,80.0,3,0.200,CargaSeca,33.3,GranelSolido
181,tarifas_historicas,valor,float64,0,0.0,13,0.867,1900000.0,20.0,6788888.0
182,tarifas_historicas,fecha,datetime64[us],0,0.0,2,0.133,2026-08-31 00:00:00,80.0,2026-08-31 00:00:00


## Paso 4 — Columnas constantes o cuasi-constantes

Filtramos del perfil anterior las columnas donde un solo valor domina 95%
o más de las filas — son columnas que, tal como están en este set, aportan
poca o ninguna señal para diferenciar un registro de otro. No se descartan
solo por esto (puede ser un hecho real y esperado, como `TYPE` siempre
`'2'` en el catálogo de clientes SAP, ya documentado) — se listan para
decidir caso por caso, no en automático.


In [4]:
columnas_cuasi_constantes = df_perfil_columnas[
    df_perfil_columnas["pct_valor_mas_frecuente"] >= 95
].sort_values("pct_valor_mas_frecuente", ascending=False)

print(f"Columnas cuasi-constantes (>=95% un solo valor): {len(columnas_cuasi_constantes)} de {len(df_perfil_columnas)}")
columnas_cuasi_constantes[["tabla", "columna", "unicos", "valor_mas_frecuente", "pct_valor_mas_frecuente"]]


Columnas cuasi-constantes (>=95% un solo valor): 25 de 184


,tabla,columna,unicos,valor_mas_frecuente,pct_valor_mas_frecuente
1,audit_log,usuarioid,1,00000000-0000-0000-0000-000000000001,100.0
7,catalogo-clientes-BUT000-organizaciones,type,1,2,100.0
15,catalogo-clientes-BUT000-organizaciones,validTo,1,9999-12-31 23:59:59,100.0
46,cotizacion_conexos,cotizacionid,1,0EE0943A-81F2-49F0-A6B8-AF5B2B04FA51,100.0
64,cotizaciones,entregadapor,1,00000000-0000-0000-0000-000000000001,100.0
88,inconsistencias,entidad,1,solicitudes,100.0
89,inconsistencias,entidadid,1,3EAB0AB6-ED6D-4983-A8EA-F9BF7406AB0A,100.0
90,inconsistencias,motivo,1,Deadline anterior a la fecha de creacion,100.0
95,lineas_solicitud,tipologia,1,CargaSeca,100.0
102,lineas_solicitud,levantecontenedordiferente,1,False,100.0


In [5]:
columnas_cuasi_constantes = df_perfil_columnas[
    df_perfil_columnas["pct_valor_mas_frecuente"] >= 95
].sort_values("pct_valor_mas_frecuente", ascending=False)

print(f"Columnas cuasi-constantes (>=95% un solo valor): {len(columnas_cuasi_constantes)} de {len(df_perfil_columnas)}")
columnas_cuasi_constantes[["tabla", "columna", "nulos", "pct_nulos", "unicos", "valor_mas_frecuente", "pct_valor_mas_frecuente"]]


Columnas cuasi-constantes (>=95% un solo valor): 25 de 184


,tabla,columna,nulos,pct_nulos,unicos,valor_mas_frecuente,pct_valor_mas_frecuente
1,audit_log,usuarioid,0,0.0,1,00000000-0000-0000-0000-000000000001,100.0
7,catalogo-clientes-BUT000-organizaciones,type,0,0.0,1,2,100.0
15,catalogo-clientes-BUT000-organizaciones,validTo,18,0.0,1,9999-12-31 23:59:59,100.0
46,cotizacion_conexos,cotizacionid,0,0.0,1,0EE0943A-81F2-49F0-A6B8-AF5B2B04FA51,100.0
64,cotizaciones,entregadapor,12,66.7,1,00000000-0000-0000-0000-000000000001,100.0
88,inconsistencias,entidad,0,0.0,1,solicitudes,100.0
89,inconsistencias,entidadid,0,0.0,1,3EAB0AB6-ED6D-4983-A8EA-F9BF7406AB0A,100.0
90,inconsistencias,motivo,0,0.0,1,Deadline anterior a la fecha de creacion,100.0
95,lineas_solicitud,tipologia,42,97.7,1,CargaSeca,100.0
102,lineas_solicitud,levantecontenedordiferente,0,0.0,1,False,100.0
